In [1]:
import pandas as pd
import re
import string
import nltk
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
import ssl

In [2]:
# Downloaded the necessary NLTK data files
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/tld/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/tld/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/tld/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/tld/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
# Load the dataset
df = pd.read_csv('/Users/tld/Desktop/CS NLP Project_v2/Pre-processed Data/fake reviews dataset_final_v3.csv')

In [4]:
df.head()

,label,text
0,1,These are good canned asparagus. I used to ge...
1,1,"I didn't buy this particular one, I bought the..."
2,0,Supposedly great as a chew toy. The only probl...
3,1,I wanted to try a new coffee company and found...
4,1,"If you don't like ginger, you're not going to ..."


### Step 1:Lowercasing

In [5]:
df['text'] = df['text'].str.lower()
df.head()

,label,text
0,1,these are good canned asparagus. i used to ge...
1,1,"i didn't buy this particular one, i bought the..."
2,0,supposedly great as a chew toy. the only probl...
3,1,i wanted to try a new coffee company and found...
4,1,"if you don't like ginger, you're not going to ..."


### Step 2: Removing HTML tags

In [6]:
df['text'] = df['text'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
df.head()

/var/folders/68/t3_092y94f30ps5gh98jhmk00000gp/T/ipykernel_2045/312075014.py:1: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  df['text'] = df['text'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())


,label,text
0,1,these are good canned asparagus. i used to ge...
1,1,"i didn't buy this particular one, i bought the..."
2,0,supposedly great as a chew toy. the only probl...
3,1,i wanted to try a new coffee company and found...
4,1,"if you don't like ginger, you're not going to ..."


### Step 3: Removing URL tags

In [7]:
def remove_urls(text):
    url_pattern = re.compile(r'http\S+|www\S+|https\S+')
    return url_pattern.sub(r'', text)

df['text'] = df['text'].apply(remove_urls)
df.head()

,label,text
0,1,these are good canned asparagus. i used to ge...
1,1,"i didn't buy this particular one, i bought the..."
2,0,supposedly great as a chew toy. the only probl...
3,1,i wanted to try a new coffee company and found...
4,1,"if you don't like ginger, you're not going to ..."


### Step 4: Removing Punctuation

In [8]:
df['text'] = df['text'].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))
df.head()

,label,text
0,1,these are good canned asparagus i used to get...
1,1,i didnt buy this particular one i bought the b...
2,0,supposedly great as a chew toy the only proble...
3,1,i wanted to try a new coffee company and found...
4,1,if you dont like ginger youre not going to lik...


### Step 5: Removing Special Characters

In [9]:
df['text'] = df['text'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))
df.head()

,label,text
0,1,these are good canned asparagus i used to get...
1,1,i didnt buy this particular one i bought the b...
2,0,supposedly great as a chew toy the only proble...
3,1,i wanted to try a new coffee company and found...
4,1,if you dont like ginger youre not going to lik...


### Step 6: Tokenization

In [10]:
df['text'] = df['text'].apply(word_tokenize)
df.head()

,label,text
0,1,"[these, are, good, canned, asparagus, i, used,..."
1,1,"[i, didnt, buy, this, particular, one, i, boug..."
2,0,"[supposedly, great, as, a, chew, toy, the, onl..."
3,1,"[i, wanted, to, try, a, new, coffee, company, ..."
4,1,"[if, you, dont, like, ginger, youre, not, goin..."


### Step 7: Stopwords Removal

In [11]:
stop_words = set(stopwords.words('english'))
df['text'] = df['text'].apply(lambda x: [word for word in x if word not in stop_words])
df.head()

,label,text
0,1,"[good, canned, asparagus, used, get, canned, a..."
1,1,"[didnt, buy, particular, one, bought, bigger, ..."
2,0,"[supposedly, great, chew, toy, problem, kind, ..."
3,1,"[wanted, try, new, coffee, company, found, bro..."
4,1,"[dont, like, ginger, youre, going, like, gold,..."


### Step 8: Lemmatization

In [12]:
lemmatizer = WordNetLemmatizer()
df['text'] = df['text'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
df.head()

,label,text
0,1,"[good, canned, asparagus, used, get, canned, a..."
1,1,"[didnt, buy, particular, one, bought, bigger, ..."
2,0,"[supposedly, great, chew, toy, problem, kind, ..."
3,1,"[wanted, try, new, coffee, company, found, bro..."
4,1,"[dont, like, ginger, youre, going, like, gold,..."


### Step 9: Removing Extra Whitespaces

In [13]:
df['text'] = df['text'].apply(lambda x: ' '.join(x).strip())
df.head()

,label,text
0,1,good canned asparagus used get canned asparagu...
1,1,didnt buy particular one bought bigger one sal...
2,0,supposedly great chew toy problem kind hard pu...
3,1,wanted try new coffee company found brooklyn c...
4,1,dont like ginger youre going like gold kilis g...


### Step 10: Handling Emojis or Emoticons

In [14]:
def remove_emojis(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F700-\U0001F77F"  # alchemical symbols
                           u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
                           u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
                           u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
                           u"\U0001FA00-\U0001FA6F"  # Chess Symbols
                           u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

df['text'] = df['text'].apply(remove_emojis)
df.head()

,label,text
0,1,good canned asparagus used get canned asparagu...
1,1,didnt buy particular one bought bigger one sal...
2,0,supposedly great chew toy problem kind hard pu...
3,1,wanted try new coffee company found brooklyn c...
4,1,dont like ginger youre going like gold kilis g...


### Final Pre-Processed file

In [15]:
df.to_csv('fake reviews dataset_final_text_preprocessed_v3.csv')

In [22]:
df

,label,text
0,1,good canned asparagus used get canned asparagu...
1,1,didnt buy particular one bought bigger one sal...
2,0,supposedly great chew toy problem kind hard pu...
3,1,wanted try new coffee company found brooklyn c...
4,1,dont like ginger youre going like gold kilis g...
...,...,...
884496,1,love garden eatin chip favorite variety hot dp...
884497,1,maybe name burned brain reading 50 shade grey ...
884498,1,little guy born eaten price even shipping beat...
884499,0,one best purchase ive made recently customizab...
